# 准备数据

In [1]:
# 导入模块
from datetime import datetime, date
import json
import os
import shelve
from functools import lru_cache
from pathlib import Path

import dbm
import polars as pl
from tqdm import tqdm

from vnpy.trader.database import DB_TZ
from vnpy.trader.datafeed import get_datafeed
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import HistoryRequest
from vnpy.alpha import AlphaLab, logger
from mytest.tushare_client import TushareClient, vt_to_ts_code


In [2]:
# 配置下载参数
task_name = "csi300"
index_symbol = "000300.SSE"
index_ts_code = vt_to_ts_code(index_symbol)

end_dt = datetime.now(DB_TZ)
end_date = end_dt.strftime("%Y-%m-%d")
logger.info(f"默认更新至 {end_date}")


def _normalize_listing_date(value: datetime | date | str | None) -> datetime | None:
    """将 TuShare 返回的上市日期规范为 DB_TZ 时区"""
    if value is None:
        return None

    if isinstance(value, datetime):
        dt_obj = value
    elif isinstance(value, date):
        dt_obj = datetime.combine(value, datetime.min.time())
    elif isinstance(value, str):
        for fmt in ("%Y-%m-%d", "%Y%m%d"):
            try:
                dt_obj = datetime.strptime(value, fmt)
                break
            except ValueError:
                continue
        else:
            return None
    else:
        return None

    if dt_obj.tzinfo is None:
        return dt_obj.replace(tzinfo=DB_TZ)

    return dt_obj.astimezone(DB_TZ)


DEFAULT_INDEX_START_DT = datetime(2007, 1, 1).replace(tzinfo=DB_TZ)
index_start_dt = DEFAULT_INDEX_START_DT
start_date = index_start_dt.strftime("%Y-%m-%d")

VT_SETTING_PATH = Path.home() / ".vntrader" / "vt_setting.json"


def _load_ts_token() -> str | None:
    """优先读取环境变量，其次读取 .vntrader 配置中的 TuShare Token。"""
    env_token = os.getenv("TS_TOKEN")
    if env_token:
        return env_token

    try:
        content = VT_SETTING_PATH.read_text(encoding="utf-8")
    except FileNotFoundError:
        logger.warning(f"未找到 {VT_SETTING_PATH}，请手动配置 TuShare Token")
        return None
    except OSError as exc:  # noqa: BLE001
        logger.error(f"读取 {VT_SETTING_PATH} 失败: {exc}")
        return None

    try:
        settings = json.loads(content)
    except json.JSONDecodeError as exc:
        logger.error(f"解析 {VT_SETTING_PATH} 失败: {exc}")
        return None

    token = settings.get("datafeed.password") or settings.get("TS_TOKEN")
    if not token:
        logger.warning(f"{VT_SETTING_PATH} 中缺少 TuShare Token 配置")
        return None

    return token


TS_TOKEN = _load_ts_token()


2025-09-30 00:05:57 默认更新至 2025-09-30


In [3]:
# 创建投研实验室
lab = AlphaLab(f"./lab/{task_name}")  # 指定数据文件夹

In [4]:
# 初始化数据服务（datafeed 及 TuShare 客户端）
datafeed = get_datafeed()
datafeed.init()

tushare_client = TushareClient(token=TS_TOKEN)

In [5]:
# 确认指数上市日期
listing_resolved = False
try:
    df_index = tushare_client._pro.index_basic(ts_code=index_ts_code)
    if df_index is not None and not df_index.empty:
        listing_candidate = None
        for column in ("list_date", "publish_date", "base_date"):
            listing_candidate = _normalize_listing_date(df_index.iloc[0].get(column))
            if listing_candidate:
                break
        if listing_candidate and listing_candidate <= end_dt:
            index_start_dt = listing_candidate
            listing_resolved = True
except Exception as exc:  # noqa: BLE001
    logger.error(f"读取指数上市日期失败: {exc}")

if not listing_resolved:
    index_start_dt = DEFAULT_INDEX_START_DT
    logger.warning("指数上市日期缺失，使用 2007-01-01 作为起点")

start_date = index_start_dt.strftime("%Y-%m-%d")
logger.info(f"指数历史起始日期 {start_date}")


2025-09-30 00:05:59 指数历史起始日期 2005-04-08


In [7]:
# 增量下载指数成分股（TuShare）
# 规范化：给 shelve 一个“文件基名”（不要传目录）
component_dir = Path(lab.component_path)
component_dir.mkdir(parents=True, exist_ok=True)
component_base = component_dir / f"{index_symbol}.shelve"  # 自用缓存文件
component_store_base = component_dir / index_symbol  # AlphaLab 默认路径


def _cleanup_component_store(base_path: Path) -> None:
    """清理可能残留的 shelve/dbm 文件，避免类型识别失败。"""
    candidates = [
        base_path,
        Path(f"{base_path}.db"),
        Path(f"{base_path}.dat"),
        Path(f"{base_path}.dir"),
        Path(f"{base_path}.bak"),
        Path(f"{base_path}.pag"),
    ]
    for candidate in candidates:
        try:
            if candidate.exists():
                candidate.unlink()
        except Exception as exc:  # noqa: BLE001
            logger.error(f"删除旧组件文件 {candidate} 失败: {exc}")


start_d = datetime.strptime(start_date, "%Y-%m-%d").date()
end_d = datetime.strptime(end_date, "%Y-%m-%d").date()

# 探测已有库（None=不存在；""=损坏/不可识别；其它=可读）
which = dbm.whichdb(str(component_base))

if which not in (None, ""):
    # 有可读库 → 找到已保存的最大日期键
    with shelve.open(str(component_base), flag="r") as db:
        latest_key = None
        for k in db:  # 避免 db.keys() 物化
            if (latest_key is None) or (k > latest_key):
                latest_key = k
        if latest_key:
            latest_saved = datetime.strptime(latest_key, "%Y-%m-%d").date()
            fetch_start = max(latest_saved, start_d)
        else:
            fetch_start = start_d
else:
    # 首次或不可识别 → 全量
    fetch_start = start_d

if fetch_start > end_d:
    # 覆盖，无需更新
    pass
else:
    component_map = tushare_client.fetch_index_components(
        index_symbol,
        datetime.combine(fetch_start, datetime.min.time()),
        datetime.combine(end_d, datetime.min.time()),
        expand_daily=True,
    )

    if component_map:
        index_components: dict[str, list[str]] = {}
        for dt_obj, vt_syms in component_map.items():
            if isinstance(dt_obj, datetime):
                base_date = dt_obj.date()
            elif isinstance(dt_obj, date):
                base_date = dt_obj
            else:
                base_date = datetime.strptime(str(dt_obj), "%Y-%m-%d").date()
            index_components[base_date.strftime("%Y-%m-%d")] = sorted(set(vt_syms))

        try:
            lab.save_component_data(index_symbol, index_components)
        except Exception as exc:  # noqa: BLE001
            logger.error(f"保存指数成分数据失败，将重建库: {exc}")
            _cleanup_component_store(component_store_base)
            lab.save_component_data(index_symbol, index_components)

        # 清空缓存以确保后续读取最新成分
        lab.load_component_data.cache_clear()

        # 同步更新 shelve（存在则覆盖，不存在则创建）
        with shelve.open(str(component_base), flag="c", writeback=False) as db:
            for k, v in index_components.items():
                db[k] = v
    else:
        logger.warning(
            f"TuShare 未返回 {index_symbol} {fetch_start:%Y-%m-%d}~{end_d:%Y-%m-%d} 的成分数据"
        )


2025-09-30 00:06:22 TuShare index_weight 空结果: 000300.SSE 20250930~20250930
2025-09-30 00:06:22 TuShare 未返回 000300.SSE 2025-09-30~2025-09-30 的成分数据
2025-09-30 00:06:22 TuShare 未返回 000300.SSE 2025-09-30~2025-09-30 的成分数据


In [8]:
# 加载指数成分股代码
component_symbols = lab.load_component_symbols(index_symbol, start_date, end_date)
print(len(component_symbols))
component_symbols[:10]

324


['002709.SZSE',
 '000538.SZSE',
 '601825.SSE',
 '601229.SSE',
 '601808.SSE',
 '601169.SSE',
 '688012.SSE',
 '605117.SSE',
 '002050.SZSE',
 '600436.SSE']

In [9]:
# 转换时间格式并确认已有数据范围
start = index_start_dt
end = end_dt


@lru_cache(maxsize=None)
def get_symbol_start(vt_symbol: str) -> datetime:
    """查询单个合约的上市日期"""
    if vt_symbol == index_symbol:
        return start

    ts_code = vt_to_ts_code(vt_symbol)
    try:
        df = tushare_client._pro.stock_basic(
            ts_code=ts_code,
            fields="ts_code,list_date,list_status,delist_date",
        )
    except Exception as exc:  # noqa: BLE001
        logger.error(f"获取 {vt_symbol} 上市日期失败: {exc}")
        return start

    listing_dt = None
    if df is not None and not df.empty:
        record = df.iloc[0]
        listing_value = record.get("list_date")
        listing_dt = _normalize_listing_date(listing_value)

    if not listing_dt:
        logger.warning(f"{vt_symbol} 缺少上市日期信息，退回指数起点 {start:%Y-%m-%d}")
        return start

    if listing_dt > end:
        logger.warning(
            f"{vt_symbol} 上市日期 {listing_dt:%Y-%m-%d} 晚于目标结束时间，跳过"
        )
        return end

    return listing_dt


def resolve_history_start(
    vt_symbol: str,
    interval: Interval,
    default_start: datetime,
) -> tuple[datetime, datetime | None]:
    """根据本地缓存决定增量下载起点"""
    if interval == Interval.DAILY:
        file_path = lab.daily_path.joinpath(f"{vt_symbol}.parquet")
    elif interval == Interval.MINUTE:
        file_path = lab.minute_path.joinpath(f"{vt_symbol}.parquet")
    else:
        return default_start, None

    if not file_path.exists():
        return default_start, None

    try:
        df = pl.read_parquet(file_path, columns=["datetime"])
    except Exception as exc:  # noqa: BLE001
        logger.error(f"读取本地 {vt_symbol} {interval.name} 数据失败: {exc}")
        return default_start, None

    if df.is_empty():
        return default_start, None

    stats = df.select(
        pl.min("datetime").alias("min_dt"),
        pl.max("datetime").alias("max_dt"),
    ).to_dict(as_series=False)

    earliest = stats["min_dt"][0]
    latest = stats["max_dt"][0]

    def ensure_db_tz(dt_obj: datetime | str | None) -> datetime | None:
        if dt_obj is None:
            return None
        if isinstance(dt_obj, str):
            dt_obj = datetime.fromisoformat(dt_obj)
        if dt_obj.tzinfo is None:
            return dt_obj.replace(tzinfo=DB_TZ)
        return dt_obj.astimezone(DB_TZ)

    earliest_dt = ensure_db_tz(earliest)
    latest_dt = ensure_db_tz(latest)

    if earliest_dt and default_start < earliest_dt:
        logger.info(
            f"{vt_symbol} 本地 {interval.name} 数据起点为 {earliest_dt:%Y-%m-%d %H:%M}，将回补更早区间"
        )
        return default_start, latest_dt

    if not latest_dt:
        return default_start, None

    request_start = max(latest_dt, default_start)
    return request_start, latest_dt


# 筛选成分股，不要把指数本身漏掉
task_symbols = component_symbols + [index_symbol]

for vt_symbol in tqdm(task_symbols, desc="增量下载K线"):
    symbol, exchange_str = vt_symbol.split(".")
    base_start = get_symbol_start(vt_symbol)

    if base_start >= end:
        logger.info(f"{vt_symbol} 上市日期晚于目标结束日期，无需下载")
        continue

    for interval in (Interval.DAILY, Interval.MINUTE):
        request_start, latest_dt = resolve_history_start(
            vt_symbol, interval, base_start
        )

        if latest_dt and latest_dt >= end:
            logger.info(
                f"{vt_symbol} {interval.name} 数据已覆盖至 {latest_dt:%Y-%m-%d %H:%M}，跳过"
            )
            continue

        req = HistoryRequest(
            symbol, Exchange(exchange_str), request_start, end, interval
        )
        bars = datafeed.query_bar_history(req)

        if bars:
            lab.save_bar_data(bars)
            logger.info(
                f"写入 {vt_symbol} {interval.name} {len(bars)} 条数据，起始 {request_start:%Y-%m-%d %H:%M}"
            )
        else:
            logger.warning(f"增量下载 {vt_symbol} {interval.name} 未返回数据")


增量下载K线:   0%|          | 0/325 [00:00<?, ?it/s]

2025-09-30 00:06:38 写入 002709.SZSE DAILY 226 条数据，起始 2024-10-30 00:00


增量下载K线:   0%|          | 0/325 [00:03<?, ?it/s]

抱歉，您每分钟最多访问该接口2次，权限的具体详情访问：https://tushare.pro/document/1?doc_id=108。
抱歉，您每分钟最多访问该接口2次，权限的具体详情访问：https://tushare.pro/document/1?doc_id=108。
抱歉，您每分钟最多访问该接口2次，权限的具体详情访问：https://tushare.pro/document/1?doc_id=108。


OSError: ERROR.

In [ ]:
# 添加回测参数配置
for vt_symbol in component_symbols:
    lab.add_contract_setting(
        vt_symbol,
        long_rate=5 / 10000,
        short_rate=10 / 10000,
        size=1,
        pricetick=0.0001,
    )